# Geocode Drinking Water Systems via EPA FRS

The `drinking water_*.csv` files (from EPA's SDWIS public water system search) have no
latitude/longitude columns -- just `PWSId`, `PWSName`, county, population served, etc. This
notebook joins each `PWSId` against EPA's **Facility Registry Service (FRS)** via the Envirofacts
`efservice` REST API to recover coordinates, in two steps per system:

1. `FRS_PROGRAM_FACILITY` -- look up the FRS `REGISTRY_ID` linked to a given SDWIS `PWSId`.
2. `FRS_FACILITY_SITE` -- look up `LATITUDE83`/`LONGITUDE83` for that `REGISTRY_ID`.

Not every public water system is registered in FRS with a geocoded site (this is especially true
for small transient systems -- individual campgrounds, churches, stores with their own well) --
those will come back with no coordinates and get flagged, not silently dropped.

**Run this on VICTOR**, not in a restricted sandbox -- `data.epa.gov` needs to be reachable.

Output: `drinking_water_<county>_geocoded.csv` per input file, with `Latitude83`/`Longitude83`
columns added. `infrastructure_exposure.ipynb`'s loader already recognizes those column names
(case-insensitive) via `LAT_KEYWORDS`/`LON_KEYWORDS`, so once these are written, just point
`LOCAL_DRINKING_WATER_PATH` at the `_geocoded.csv` files instead of the originals -- no other
notebook changes needed.

In [ ]:
import os
import time
import requests
import pandas as pd

EFSERVICE_BASE = "https://data.epa.gov/efservice"
REQUEST_TIMEOUT = 90  # data.epa.gov/efservice is slow -- 30s wasn't enough, confirmed via testing on VICTOR
REQUEST_DELAY_S = 0.2  # be polite to the API between calls
CACHE_PATH = "drinking_water_frs_coords_cache.csv"

DRINKING_WATER_SOURCES = {
    "Hood River": "drinking water_hood_river_8_10_2026.csv",
    "Clackamas": "drinking water_clackamas_8_10_2026.csv",
}

## Section 1 -- Load the source PWS lists

In [ ]:
dw_frames = []
for county, path in DRINKING_WATER_SOURCES.items():
    df = pd.read_csv(path)
    df["SourceCounty"] = county
    dw_frames.append(df)

drinking_water_df = pd.concat(dw_frames, ignore_index=True)
print(f"Loaded {len(drinking_water_df)} public water systems ({drinking_water_df['PWSId'].nunique()} unique PWSIds)")
drinking_water_df.head()

## Section 2 -- Diagnostic: test the API against one real PWSId

Run this cell **first** and read the printed output before running the bulk join in Section 3.

`FRS_PROGRAM_FACILITY` and `FRS_FACILITY_SITE` are the table/column names as documented by EPA
Envirofacts, but I couldn't test this query live (this environment's network egress blocks
`data.epa.gov`). If the status code isn't 200, or the JSON below doesn't contain a `REGISTRY_ID`
field, paste the printed output back and I'll adjust the query -- same as we had to do for the
roads/transmission/facilities APIs earlier.

In [ ]:
test_pwsid = drinking_water_df["PWSId"].iloc[0]
test_url = f"{EFSERVICE_BASE}/FRS_PROGRAM_FACILITY/PGM_SYS_ACRNM/=/SDWIS/PGM_SYS_ID/=/{test_pwsid}/JSON"

print("Testing:", test_url)
resp = requests.get(test_url, timeout=REQUEST_TIMEOUT)
print("status:", resp.status_code)
print(resp.text[:3000])

In [ ]:
# If Section 2 returned a REGISTRY_ID, sanity-check the second lookup too.
test_records = resp.json()
if test_records:
    test_registry_id = test_records[0].get("REGISTRY_ID")
    site_url = f"{EFSERVICE_BASE}/FRS_FACILITY_SITE/REGISTRY_ID/=/{test_registry_id}/JSON"
    print("Testing:", site_url)
    site_resp = requests.get(site_url, timeout=REQUEST_TIMEOUT)
    print("status:", site_resp.status_code)
    print(site_resp.text[:3000])
else:
    print(f"No FRS_PROGRAM_FACILITY record found for PWSId {test_pwsid} -- this system may not be in FRS at all.")

## Section 3 -- Bulk join (resumable)

Two API calls per unique `PWSId` (~266 systems total). Results are cached to
`drinking_water_frs_coords_cache.csv` after every 20 lookups, so if the kernel dies or the API
rate-limits you partway through, re-running this cell picks up where it left off instead of
starting over -- same pattern as the Tephra2 sweep notebooks.

In [ ]:
def fetch_json(url):
    resp = requests.get(url, timeout=REQUEST_TIMEOUT)
    resp.raise_for_status()
    return resp.json()


def get_registry_id(pwsid):
    url = f"{EFSERVICE_BASE}/FRS_PROGRAM_FACILITY/PGM_SYS_ACRNM/=/SDWIS/PGM_SYS_ID/=/{pwsid}/JSON"
    records = fetch_json(url)
    return records[0].get("REGISTRY_ID") if records else None


def get_coords(registry_id):
    url = f"{EFSERVICE_BASE}/FRS_FACILITY_SITE/REGISTRY_ID/=/{registry_id}/JSON"
    records = fetch_json(url)
    if not records:
        return None, None
    rec = records[0]
    return rec.get("LATITUDE83"), rec.get("LONGITUDE83")


if os.path.exists(CACHE_PATH):
    coords_df = pd.read_csv(CACHE_PATH)
    done_ids = set(coords_df["PWSId"])
    print(f"Resuming: {len(done_ids)} PWSIds already geocoded in cache")
    rows = coords_df.to_dict("records")
else:
    done_ids = set()
    rows = []

unique_pwsids = drinking_water_df["PWSId"].unique()
remaining = [p for p in unique_pwsids if p not in done_ids]
print(f"{len(remaining)} PWSIds left to geocode")

for i, pwsid in enumerate(remaining):
    try:
        registry_id = get_registry_id(pwsid)
        if registry_id is None:
            rows.append({"PWSId": pwsid, "RegistryID": None, "Latitude83": None, "Longitude83": None, "Status": "not_in_frs"})
        else:
            lat, lon = get_coords(registry_id)
            if lat is None or lon is None:
                rows.append({"PWSId": pwsid, "RegistryID": registry_id, "Latitude83": None, "Longitude83": None, "Status": "no_coords"})
            else:
                rows.append({"PWSId": pwsid, "RegistryID": registry_id, "Latitude83": lat, "Longitude83": lon, "Status": "ok"})
    except Exception as e:
        rows.append({"PWSId": pwsid, "RegistryID": None, "Latitude83": None, "Longitude83": None, "Status": f"error: {e}"})

    if (i + 1) % 20 == 0 or (i + 1) == len(remaining):
        pd.DataFrame(rows).to_csv(CACHE_PATH, index=False)
        print(f"  {i + 1}/{len(remaining)} done, cached to {CACHE_PATH}")

    time.sleep(REQUEST_DELAY_S)

coords_df = pd.DataFrame(rows)
coords_df.to_csv(CACHE_PATH, index=False)

print("\nGeocoding results:")
print(coords_df["Status"].value_counts())

## Section 4 -- Merge back and write geocoded output files

In [ ]:
merged = drinking_water_df.merge(
    coords_df[["PWSId", "Latitude83", "Longitude83", "Status"]], on="PWSId", how="left"
)

matched = merged[merged["Status"] == "ok"]
unmatched = merged[merged["Status"] != "ok"]
print(f"{len(matched)} of {len(merged)} water systems geocoded successfully")

if len(unmatched):
    print(f"\n{len(unmatched)} systems could not be geocoded via FRS (not registered, or no site coordinates on file):")
    print(unmatched[["PWSName", "PWSId", "SourceCounty", "Status"]].to_string(index=False))

for county in merged["SourceCounty"].unique():
    out_path = f"drinking_water_{county.lower().replace(' ', '_')}_geocoded.csv"
    merged[merged["SourceCounty"] == county].to_csv(out_path, index=False)
    print(f"\nWrote {out_path} ({(merged['SourceCounty'] == county).sum()} rows)")

## Section 5 -- Manual fallback for the ~30 systems near Mt. Hood

The FRS join in Sections 2-4 came back with 0/265 real matches (the `PGM_SYS_ACRNM` value and
table structure guessed from memory didn't line up with the live API), and a follow-up attempt
at ECHO's `get_dfr` REST service also 404'd. Rather than keep guessing at undocumented API
internals, this does the join by hand -- but only for the water systems that could plausibly
fall within the 5 km hazard buffers around Rhododendron, Parkdale, or Govt. Camp. Everything else
in the two county-wide CSVs (Milwaukie, Wilsonville, Molalla, Canby, ...) is tens of km away and
geometrically irrelevant to `infrastructure_exposure.ipynb` regardless of its coordinates, so
there's no need to geocode it.

**To fill in a coordinate**: open that system's own `DFR URL` value from the source CSV (or just
swap the `PWSId` into `https://echo.epa.gov/detailed-facility-report?fid=<PWSId>&sys=SDWIS`), find
the Latitude/Longitude on that page, and fill in the tuple below. Leave any system `None` if it
isn't listed on ECHO (small transient systems sometimes aren't) -- it'll just be skipped, printed
in the "still missing" list, and excluded from the output, not silently left with garbage coordinates.

In [ ]:
# PWSId -> (latitude, longitude). Fill these in from each system's ECHO Detailed Facility Report.
MANUAL_COORDS = {
    # -- Hood River County --
    "OR4100612": None,  # PARKDALE WATER COMPANY INC
    "OR4191163": None,  # COOPER SPUR MOUNTAIN RESORT INN
    "OR4191165": None,  # COOPER SPUR SKI AREA
    "OR4195046": None,  # MT HOOD MEADOWS HRM & NORDIC
    "OR4191167": None,  # MT HOOD MEADOWS SPRING

    # -- Clackamas County --
    "OR4100336": None,  # GOVERNMENT CAMP WATER SYSTEM
    "OR4100702": None,  # RHODODENDRON WTR ASSOC
    "OR4100937": None,  # WELCHES WATER COMPANY
    "OR4193546": None,  # WELCHES MOUNTAIN PROPERTIES
    "OR4101241": None,  # ZIG ZAG WATER CO-OP
    "OR4101159": None,  # ZIG ZAG VILLAGE WATER SYSTEM
    "OR4195216": None,  # ZIG ZAG MOUNTAIN STORE
    "OR4106250": None,  # ZIG ZAG MTN FARM
    "OR4193574": None,  # ZIG ZAG INN
    "OR4100639": None,  # TIMBERLINE RIM WATER CO INC
    "OR4100143": None,  # BRIGHTWOOD WATER WORKS
    "OR4105395": None,  # NORTH BRIGHTWOOD IMPROV ASSN
    "OR4105429": None,  # ARRAH WANNA ESTATES WS
    "OR4101265": None,  # ARRAH WANNA WATER COMPANY
    "OR4190034": None,  # CAMP ARRAH WANNA
    "OR4190063": None,  # MT HOOD KIWANIS CAMP
    "OR4194841": None,  # MT HOOD INN
    "OR4194548": None,  # MT HOOD RV VILLAGE
    "OR4193635": None,  # MT HOOD BREWING COMPANY
    "OR4192639": None,  # USFS TRILLIUM LAKE CG
    "OR4194843": None,  # USFS TOLLGATE CG
    "OR4192641": None,  # USFS TIMBERLINE LODGE 1
    "OR4193557": None,  # MARKUM INN
    "OR4193457": None,  # SUMMIT CAFE
    "OR4100637": None,  # SALMON RIVER PARK WTR IMP DIST
    "OR4100936": None,  # SALMON VALLEY WATER COMPANY
}

In [ ]:
# Mt. Hood region sanity bounds -- catches a mis-typed coordinate before it silently corrupts the layer.
LAT_BOUNDS = (44.9, 45.9)
LON_BOUNDS = (-122.1, -121.2)

filled = {pwsid: coords for pwsid, coords in MANUAL_COORDS.items() if coords is not None}
missing = [pwsid for pwsid, coords in MANUAL_COORDS.items() if coords is None]

for pwsid, (lat, lon) in filled.items():
    if not (LAT_BOUNDS[0] <= lat <= LAT_BOUNDS[1] and LON_BOUNDS[0] <= lon <= LON_BOUNDS[1]):
        raise ValueError(
            f"{pwsid}: ({lat}, {lon}) is outside the expected Mt. Hood region {LAT_BOUNDS}/{LON_BOUNDS} "
            f"-- check for a typo (e.g. a missing minus sign on longitude) before continuing."
        )

print(f"{len(filled)} of {len(MANUAL_COORDS)} shortlisted systems have coordinates")
if missing:
    name_lookup = drinking_water_df.set_index("PWSId")["PWSName"].to_dict()
    print("Still missing:")
    for pwsid in missing:
        print(f"  {pwsid}  {name_lookup.get(pwsid, '?')}")

manual_df = pd.DataFrame(
    [{"PWSId": pwsid, "Latitude83": lat, "Longitude83": lon} for pwsid, (lat, lon) in filled.items()]
)
merged_manual = drinking_water_df.merge(manual_df, on="PWSId", how="inner")
print(f"\n{len(merged_manual)} systems geocoded and ready to write out")

for county in merged_manual["SourceCounty"].unique():
    out_path = f"drinking_water_{county.lower().replace(' ', '_')}_geocoded.csv"
    merged_manual[merged_manual["SourceCounty"] == county].to_csv(out_path, index=False)
    print(f"Wrote {out_path} ({(merged_manual['SourceCounty'] == county).sum()} rows)")

## Next step

In `infrastructure_exposure.ipynb`, change:

```python
LOCAL_DRINKING_WATER_PATH = ["drinking water_clackamas_8_10_2026.csv", "drinking water_hood_river_8_10_2026.csv"]
```

to:

```python
LOCAL_DRINKING_WATER_PATH = ["drinking_water_clackamas_geocoded.csv", "drinking_water_hood_river_geocoded.csv"]
```

No other changes needed -- `load_local_vector()` already matches `Latitude83`/`Longitude83`
case-insensitively via `LAT_KEYWORDS`/`LON_KEYWORDS`, and rows with no coordinates (`Status !=
"ok"`) will simply be dropped when building point geometry, the same as any other row missing
lat/lon.